# Пет-проект формирования реестра для оплаты государственной пошлины


Данный проект направлен на автоматизацию формирования реестра для оплаты государственной пошлины при запросе у пользователя минимальных индивидуальных данных для формирования в дальнейшем Платежного поручения.

In [1]:
# Импортируем библиотеку
import pandas as pd

Загружаем необходимые шаблоны и реестры для заполнения файла для оплаты

In [2]:
# Загружаем пустой шаблон для оплаты ГП
file_path_sample = r"C:\Users\Администратор\Desktop\Оплата ГП\Шаблон реестра на оплату ГП.csv"
sample = pd.read_csv(file_path_sample, sep=';', encoding="utf-8")
display(sample.head())

# Загружаем реестр с неизменяемыми данными
file_path_details= r"C:\Users\Администратор\Desktop\Оплата ГП\Неизменные реквизиты.csv"
details = pd.read_csv(file_path_details, sep=';', encoding="utf-8", dtype={'БИК банка получателя': str})
display(details.head())

# Загружаем справочник судебных участков
file_path_directory = r"C:\Users\Администратор\Desktop\Оплата ГП\Справочник судебных участков.csv"
directory = pd.read_csv(file_path_directory, sep=';', encoding="utf-8", dtype={'ОКТМО': str})
display(directory.head())

,Дата п/п,Категория затрат,Наименование плательщика (Организация),ИНН Плательщика,КПП Плательщика,"Банковский счет Организации (Наименование, номер счета, банк)",Наименование Получателя,ИНН Получателя,КПП Получателя,"Счет Получателя (Наименование, номер счета, банк)",...,сумма платежа,Назначение платежа0,Назначение платежа1,Назначение платежа2,Назначение платежа3,Назначение платежа4,Назначение платежа5,Подразделение,БИК банка получателя,Основание платежа


,Категория затрат,Наименование плательщика (Организация),ИНН Плательщика,КПП Плательщика,"Банковский счет Организации (Наименование, номер счета, банк)",Наименование Получателя,ИНН Получателя,КПП Получателя,"Счет Получателя (Наименование, номер счета, банк)",Статья движения денежных средств,...,КБК,налоговый период,номер документа,дата документа,Назначение платежа0,Назначение платежа2,Назначение платежа4,Подразделение,БИК банка получателя,Основание платежа
0,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,1.2.13.04 Госпошлина,...,18210803010011050110,0,0,0,"Госпошлина по делам, рассматриваемым в судах о...",Договор №,от,НОВОСИБИРСК ОЮС,017003983,0


,Регион,Классификационный_код,Наименование_судебного_участка,Адрес_участка,ОКТМО
0,22 - Алтайский край,22MS0001,Судебный участок № 1 Центрального района г. Ба...,"656031, край Алтайский, г. Барнаул, ул Силикат...",01701000
1,22 - Алтайский край,22MS0002,Судебный участок № 2 Центрального района г. Ба...,"656031, край Алтайский, г. Барнаул, ул Силикат...",01701000
2,22 - Алтайский край,22MS0003,Судебный участок № 3 Центрального района г. Ба...,"656031, край Алтайский, г. Барнаул, ул Силикат...",01701000
3,22 - Алтайский край,22MS0004,Судебный участок № 4 Центрального района г. Ба...,"656031, край Алтайский, г. Барнаул, ул Силикат...",01701000
4,22 - Алтайский край,22MS0005,Судебный участок № 5 Центрального района г. Ба...,"656031, край Алтайский, г. Барнаул, ул Силикат...",01701000


In [3]:
# Запрашиваем путь к файлу для оплаты у пользователя в формате CSV UTF-8
file_path_payment = input(r"Введите полный путь к CSV-файлу (Пример: C:\Users\Название файла.csv): ")
payment_file = pd.read_csv(file_path_payment, sep=';', encoding="utf-8")

# Запрашиваем дату оплаты
date_payment = input("Введите дату оплаты в формате дд.мм.гггг: ")

payment_file.head()

Введите полный путь к CSV-файлу (Пример: C:\Users\Название файла.csv):  C:\Users\Администратор\Desktop\Оплата ГП\Файл для оплаты ГП от 01.05.2026г.csv
Введите дату оплаты в формате дд.мм.гггг:  01.05.2026


,Номер_договора,Дата_договора,ФИО,Цена_иска,Классификационный_код,Вид_подачи
0,996543211,01.12.2025,Иванов Сергей Дмитриевич,65000,22MS0027,СП
1,667743211,24.05.2022,Смирнова Анна Викторовна,125000,19MS0033,СП
2,554322345,30.11.2024,Петров Алексей Николаевич,25300,77MS0152,СП
3,888543328,18.07.2023,Козлова Мария Павловна,310000,77MS0245,ИЗ
4,927668117,12.12.2023,Морозов Никита Андреевич,520000,19RS0004,ИЗ


При оплате государственной пошлины для исков подлежащих оценке действует норма права, содержащая расчет в зависимости от суммы.

    Выдержка из НК РФ Статья 333.19

1) при подаче искового заявления имущественного характера, административного искового заявления имущественного характера, подлежащих оценке, при цене иска:

* до 100 000 рублей - 4000 рублей;

* от 100 001 рубля до 300 000 рублей - 4000 рублей плюс 3 процента суммы, превышающей 100 000 рублей;

* от 300 001 рубля до 500 000 рублей - 10 000 рублей плюс 2,5 процента суммы, превышающей 300 000 рублей;

* от 500 001 рубля до 1 000 000 рублей - 15 000 рублей плюс 2 процента суммы, превышающей 500 000 рублей;

* от 1 000 001 рубля до 3 000 000 рублей - 25 000 рублей плюс 1 процент суммы, превышающей 1 000 000 рублей;

* от 3 000 001 рубля до 8 000 000 рублей - 45 000 рублей плюс 0,7 процента суммы, превышающей 3 000 000 рублей;

* от 8 000 001 рубля до 24 000 000 рублей - 80 000 рублей плюс 0,35 процента суммы, превышающей 8 000 000 рублей;

* от 24 000 001 рубля до 50 000 000 рублей - 136 000 рублей плюс 0,3 процента суммы, превышающей 24 000 000 рублей;

* от 50 000 001 рубля до 100 000 000 рублей - 214 000 рублей плюс 0,2 процента суммы, превышающей 50 000 000 рублей;

*  свыше 100 000 000 рублей - 314 000 рублей плюс 0,15 процента суммы, превышающей 100 000 000 рублей, но не более 900 000 рублей;

2) при подаче заявления о вынесении судебного приказа - 50 процентов размера государственной пошлины, взимаемой при подаче искового заявления имущественного характера;

In [4]:
# Создаем функцию для расчета государственной пошлины
def calculate_gp(row):
    price = row['Цена_иска']
    
    # Расчёт ГП по диапазонам
    # до 100 000 рублей - 4000 рублей
    if price <= 100000:
        gp = 4000
    # от 100 001 рубля до 300 000 рублей - 4000 рублей плюс 3 процента суммы, превышающей 100 000 рублей
    elif price <= 300000:
        gp = 4000 + (price - 100000) * 0.03
    # от 300 001 рубля до 500 000 рублей - 10 000 рублей плюс 2,5 процента суммы, превышающей 300 000 рублей
    elif price <= 500000:
        gp = 10000 + (price - 300000) * 0.025
    # от 500 001 рубля до 1 000 000 рублей - 15 000 рублей плюс 2 процента суммы, превышающей 500 000 рублей
    elif price <= 1000000:
        gp = 15000 + (price - 500000) * 0.02
    # от 1 000 001 рубля до 3 000 000 рублей - 25 000 рублей плюс 1 процент суммы, превышающей 1 000 000 рублей
    elif price <= 3000000:
        gp = 25000 + (price - 1000000) * 0.01
    # от 3 000 001 рубля до 8 000 000 рублей - 45 000 рублей плюс 0,7 процента суммы, превышающей 3 000 000 рублей
    elif price <= 8000000:
        gp = 45000 + (price - 3000000) * 0.007
    # от 8 000 001 рубля до 24 000 000 рублей - 80 000 рублей плюс 0,35 процента суммы, превышающей 8 000 000 рублей
    elif price <= 24000000:
        gp = 80000 + (price - 8000000) * 0.0035
    # от 24 000 001 рубля до 50 000 000 рублей - 136 000 рублей плюс 0,3 процента суммы, превышающей 24 000 000 рублей
    elif price <= 50000000:
        gp = 136000 + (price - 24000000) * 0.003
    # от 50 000 001 рубля до 100 000 000 рублей - 214 000 рублей плюс 0,2 процента суммы, превышающей 50 000 000 рублей
    elif price <= 100000000:
        gp = 214000 + (price - 50000000) * 0.002
    # свыше 100 000 000 рублей - 314 000 рублей плюс 0,15 процента суммы, превышающей 100 000 000 рублей, но не более 900 000 рублей
    else:
        gp = 314000 + (price - 100000000) * 0.0015
        # Ограничение максимальной суммы ГП
        gp = min(gp, 900000)

    # Применение скидки 50 % для вида подачи «Судебный Приказ»
    if row.get('Вид_подачи') == 'СП':
        gp = gp / 2

    return gp

payment_file['ГП'] = payment_file.apply(calculate_gp, axis=1)

# Добавляем ОКТМО для оплаты
payment_file = payment_file.merge(directory[['Классификационный_код', 'ОКТМО']], on = 'Классификационный_код', how = 'left' )

# Выводим полученные данные
payment_file

,Номер_договора,Дата_договора,ФИО,Цена_иска,Классификационный_код,Вид_подачи,ГП,ОКТМО
0,996543211,01.12.2025,Иванов Сергей Дмитриевич,65000,22MS0027,СП,2000.0,01716000
1,667743211,24.05.2022,Смирнова Анна Викторовна,125000,19MS0033,СП,2375.0,95635410
2,554322345,30.11.2024,Петров Алексей Николаевич,25300,77MS0152,СП,2000.0,45372000
3,888543328,18.07.2023,Козлова Мария Павловна,310000,77MS0245,ИЗ,10250.0,45914000
4,927668117,12.12.2023,Морозов Никита Андреевич,520000,19RS0004,ИЗ,15400.0,95608405
5,223457899,11.03.2020,Волкова Екатерина Романовна,30000,77MS0210,СП,2000.0,45321000


In [5]:
# Формируем реестр по шаблону - заполняем столбцы данными
sample['Код ОКАТО (ОКТМО)'] = payment_file['ОКТМО']
sample['сумма платежа'] = payment_file['ГП']
sample['Назначение платежа3'] = payment_file['Номер_договора']
sample['Назначение платежа5'] = payment_file['Дата_договора']
sample['Дата п/п'] = date_payment

for col in details.columns:
    sample[col] = details[col].values[0]

    
# Преобразуем КБК в строку ДО сохранения
sample['КБК'] = sample['КБК'].astype(str)

# Сохраняем файл 
file = input(r"Введите полный путь для сохранения и название (Пример: C:\Users\Название файла.xlsx): ")
sample.to_excel(file, index=False)    

# Выводим полученные данные для наглядности
display(sample)

Введите полный путь для сохранения и название (Пример: C:\Users\Название файла.xlsx):  C:\Users\Администратор\Desktop\Оплата ГП\Реестр оплаты 01.05.2026г.xlsx


,Дата п/п,Категория затрат,Наименование плательщика (Организация),ИНН Плательщика,КПП Плательщика,"Банковский счет Организации (Наименование, номер счета, банк)",Наименование Получателя,ИНН Получателя,КПП Получателя,"Счет Получателя (Наименование, номер счета, банк)",...,сумма платежа,Назначение платежа0,Назначение платежа1,Назначение платежа2,Назначение платежа3,Назначение платежа4,Назначение платежа5,Подразделение,БИК банка получателя,Основание платежа
0,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,2000.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,996543211,от,01.12.2025,НОВОСИБИРСК ОЮС,017003983,0
1,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,2375.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,667743211,от,24.05.2022,НОВОСИБИРСК ОЮС,017003983,0
2,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,2000.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,554322345,от,30.11.2024,НОВОСИБИРСК ОЮС,017003983,0
3,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,10250.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,888543328,от,18.07.2023,НОВОСИБИРСК ОЮС,017003983,0
4,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,15400.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,927668117,от,12.12.2023,НОВОСИБИРСК ОЮС,017003983,0
5,01.05.2026,Текущие,"ООО ""Бюро Судебного Взыскания""",2225130330,540601001,СИБИРСКИЙ БАНК ПАО СБЕРБАНК Г. НОВОСИБИРСК; 40...,Казначейство России (ФНС России),7727406020,770701001,ОКЦ № 7 ГУ Банка России по ЦФО//УФК по Тульско...,...,2000.0,"Госпошлина по делам, рассматриваемым в судах о...",NaN,Договор №,223457899,от,11.03.2020,НОВОСИБИРСК ОЮС,017003983,0


In [6]:
C:\Users\Администратор\Desktop\Оплата ГП\Файл для оплаты ГП от 01.05.2026г.csv

SyntaxError: unexpected character after line continuation character (1605828867.py, line 1)

In [ ]:
C:\Users\Администратор\Desktop\Оплата ГП\Реестр оплаты 01.05.2026г.xlsx